# 4 · Phonological normalisation of the IPA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chemvatho/kolsch-tandem/blob/main/04_normalisation/04_phonological_normalisation.ipynb)

**Pipeline stage 4 of 6.** The book's semi-phonetic orthography carries
artefacts that are *spelling conventions*, not phonological facts. We apply
three rewrite rules and then tokenise each IPA word into phonemes against the
44-symbol Kölsch inventory, producing the phoneme-segmented references used to
train the recogniser.

**Rules**
- **(a) Degemination** — a doubled consonant only marks a short preceding vowel:
  `mʊttəʁ → mʊtəʁ`
- **(b) Double vowel → long vowel** — `vɛɛdən → vɛːdən`
- **(c) Silent *Dehnungs-h* → length** — `jəsaht → jəsaːt` (only when the *h* is
  pre-consonantal or word-final; an intervocalic *h* is kept, e.g. `dɔhɪn`).

## 1 · The Kölsch phoneme inventory (44 working symbols)

In [ ]:
# === Portable setup — identical paths in VS Code, Jupyter & Google Colab ===
import os, sys
from pathlib import Path

# On Google Colab: clone the repo once (or mount Drive and point _root at it).
try:
    import google.colab  # noqa: F401
    _here = any((Path(p)/"kolsch_paths.py").exists() for p in [Path.cwd(), *Path.cwd().parents])
    if not _here and not Path("/content/kolsch-tandem/kolsch_paths.py").exists():
        os.system("git clone -q https://github.com/chemvatho/kolsch-tandem.git /content/kolsch-tandem")
except Exception:
    pass

# Repo root = the folder that contains kolsch_paths.py (found from any subfolder).
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"kolsch_paths.py").exists()),
             Path("/content/kolsch-tandem"))
sys.path.insert(0, str(_root))
from kolsch_paths import ROOT, DATA, PAGES, AUDIO, TRANS, SEG, INDEX, LEXICON, MODELS
os.chdir(ROOT)                       # any remaining relative paths resolve at the root
print("repo root:", ROOT)

In [ ]:
PHONEME_MAPPING = {
    # long vowels
    'aː','ɛː','eː','iː','oː','uː','yː','øː',
    # diphthongs (German + Kölsch falling)
    'aɪ','aʊ','ɔɪ','ɛɪ','ɔʏ','ɐʊ','ɐɥ','ɐɪ',
    # affricates
    't͡s','p͡f','t͡ʃ',
    # short vowels
    'a','ɛ','ɪ','ɔ','ʊ','ʏ','œ','ə','ɐ','e','o','i','u','y','ø',
    # fricatives
    'ʃ','ʒ','ç','χ','x','f','v','s','z','h',
    # stops + glottal
    'p','b','t','d','k','ɡ','g','ʔ',
    # nasals
    'm','n','ŋ',
    # liquids / approximants
    'l','ʁ','j','r','w','ɥ',
}
_P = sorted(PHONEME_MAPPING, key=len, reverse=True)  # longest-match first
print(len(PHONEME_MAPPING), "phoneme symbols")

## 2 · The three normalisation rules

In [ ]:
import re

LONG = {'a':'aː','ɛ':'ɛː','e':'eː','i':'iː','o':'oː','u':'uː','y':'yː','ø':'øː','œ':'øː'}
CONS = set('pbtdkɡgʔmnŋlʁjrwɥʃʒçχxfvszh')

def rule_a_degeminate(s):
    """(a) collapse doubled consonants: mʊttəʁ -> mʊtəʁ"""
    out = []
    for i, ch in enumerate(s):
        if out and ch == out[-1] and ch in CONS:
            continue
        out.append(ch)
    return "".join(out)

def rule_b_double_vowel(s):
    """(b) double vowel -> long vowel: vɛɛdən -> vɛːdən"""
    for v, lv in LONG.items():
        s = s.replace(v + v, lv)
    return s

def rule_c_dehnungs_h(s):
    """(c) silent h after a vowel, pre-consonant or word-final -> length.
       Intervocalic h is kept (often a real /h/ at a morpheme boundary)."""
    VOW = "aɛɪɔʊʏœəɐeoiuyø"
    # vowel + h + (consonant | end)  ->  long vowel
    def repl(m):
        v = m.group(1)
        return LONG.get(v, v + 'ː')
    s = re.sub(rf"([{VOW}])h(?=[{''.join(CONS)}])", repl, s)
    s = re.sub(rf"([{VOW}])h$", repl, s)
    return s

def normalise_ipa(word):
    return rule_c_dehnungs_h(rule_b_double_vowel(rule_a_degeminate(word)))

for w in ["mʊttəʁ", "vɛɛdən", "jəsaht", "dɔhɪn"]:
    print(f"{w:10s} -> {normalise_ipa(w)}")

## 3 · Longest-match phoneme tokeniser

In [ ]:
def tokenize_word(word):
    """Split a normalised IPA word into phonemes (multi-char units first)."""
    out, i = [], 0
    while i < len(word):
        for p in _P:
            if word.startswith(p, i):
                out.append(p); i += len(p); break
        else:
            i += 1   # skip stress marks / unknown chars
    return out

def tokenize_ipa(text):
    """Whole IPA string -> 'p h o n | p h o n' (phonemes space-sep, words by |)."""
    words = [" ".join(tokenize_word(normalise_ipa(w))) for w in str(text).split()]
    return " | ".join(w for w in words if w)

print(tokenize_ipa("dat əsʊ jəsaht"))

## 4 · Apply to the corpus + zero-change safety pass

Run the normalisation over your IPA column, then re-verify with a safety pass:
re-tokenising an already-normalised string must produce **zero** further
changes (idempotence).

## Kölsch pronunciation dictionary (orthography → IPA)

We use **both orthography and IPA**. The bridge is a Kölsch **pronunciation
dictionary** plus the rule-based G2P converter that generated it:

- **`kolsch_g2p.py`** — deterministic grapheme→phoneme rules (Kölsch diphthongs
  `eï→ɛɪ`, uvular `/ʁ/`, `<z>→/t͡s/`, `<w>→/v/`, `<ch>` ich-/ach-laut split,
  word-final devoicing). Converts *any* word, so it covers OOV.
- **`data/lexicon.csv`** — the dictionary itself (`kolsch, ipa, frequency`),
  generated by running the converter over the corpus vocabulary. Ships here
  built from the example; regenerate over the full corpus to reproduce the
  ~7.3k-entry lexicon.

**Lookup policy:** dictionary first (frozen, human-checkable), converter as the
fallback for words not in it.

In [ ]:
# ---- Kölsch G2P: dictionary-first, rule-based converter for OOV ----
import os, re, csv, collections, pandas as pd
from kolsch_g2p import word_to_ipa

MANIFEST = os.path.join(SEG, "manifest.csv")   # SEG, LEXICON from kolsch_paths
_APOS="'’ʼʻ`"
def _clean(w):
    w=str(w).lower()
    for a in _APOS: w=w.replace(a,"")
    return re.sub(r"[^a-zäöüßï]","",w)

def load_lexicon(path=LEXICON):
    lex={}
    if os.path.exists(path):
        for row in csv.DictReader(open(path,encoding="utf-8")):
            lex[row["kolsch"]] = row["ipa"]
    return lex
LEX = load_lexicon()

def build_lexicon(words, path=LEXICON):
    """(Re)build data/lexicon.csv from a vocabulary using the converter."""
    freq=collections.Counter(w for w in map(_clean, words) if w)
    key=lambda w: (w.translate(str.maketrans("äöüßï","aousi")), w)
    rows=[(w, word_to_ipa(w), freq[w]) for w in sorted(freq, key=key)]
    with open(path,"w",newline="",encoding="utf-8") as f:
        wr=csv.writer(f); wr.writerow(["kolsch","ipa","frequency"]); wr.writerows(rows)
    return {w:i for w,i,_ in rows}

def g2p_word(w):
    c=_clean(w)
    return LEX.get(c) or word_to_ipa(c)      # dictionary first, converter for OOV

def g2p(text):
    return " ".join(p for p in (g2p_word(w) for w in str(text).split()) if p)

if os.path.exists(MANIFEST):
    man=pd.read_csv(MANIFEST)
    # grow the dictionary with any new words, then phonemise
    LEX = build_lexicon([w for t in man["text"] for w in str(t).split()])
    man["ipa_wordform"]=man["text"].map(g2p)            # orthography -> IPA words
    man["phonetic"]=man["ipa_wordform"].map(tokenize_ipa)  # -> phoneme tokens (+rules)
    man.to_csv(MANIFEST,index=False)
    print(f"phonemised {len(man)} utterances | dictionary {len(LEX)} entries")
    print(man[["id","text","ipa_wordform","phonetic"]].head().to_string())
else:
    demo=pd.DataFrame({"text":["dat es su jesaht","ich han et jesinn"]})
    demo["ipa_wordform"]=demo["text"].map(g2p)
    demo["phonetic"]=demo["ipa_wordform"].map(tokenize_ipa)
    print("No manifest yet (run Notebook 3). G2P demo on sample text:")
    print(demo.to_string())

## Output

The `phonetic` column (`p h | p h` format) is the training label. Carry it into
the manifest produced in Notebook 3 (join on utterance id), then train in
Notebook 5. Keep the human-verified Kölsch pronunciation dictionary as the norm
and report deviations for expert judgement rather than auto-correcting.